# 00 — Baseline inference smoke test

Load the **base** instruct model (`unsloth/Llama-3.2-3B-Instruct`) via `earnings_call_research_assistant.inference` and run a few research-style earnings-call prompts.

**Kaggle**: enable GPU (T4). This notebook is a short smoke test only — no training.

Outputs here are the *before* snapshots for later base-vs-adapter comparison.

## 1. Install (Kaggle)

In [ ]:
# Detect Kaggle so we only pip-install on the hosted runtime.
from pathlib import Path
import sys

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

# Prefer the repo package when the notebook lives next to `src/`.
repo_src = Path("../src").resolve()
if repo_src.exists() and str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))
    print("added", repo_src)

In [ ]:
if IN_KAGGLE:
    %pip install -q unsloth transformers accelerate bitsandbytes pyyaml

## 2. Config

In [ ]:
from pathlib import Path
import yaml

from earnings_call_research_assistant.inference import InferenceConfig

cfg = InferenceConfig()
cfg_path = Path("../configs/default.yaml")
if cfg_path.exists():
    with cfg_path.open() as f:
        cfg = InferenceConfig.from_mapping(yaml.safe_load(f))
    print("Loaded", cfg_path)
else:
    print("No local config found; using InferenceConfig defaults.")

print(cfg)

## 3. Load base model

In [ ]:
import torch
from earnings_call_research_assistant.inference import InferenceHarness

print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

harness = InferenceHarness.from_pretrained(cfg)
print("loaded", type(harness.model).__name__)

## 4. Research-style smoke prompts

These are *ungrounded* prompts (no transcript attached). After Phase 1 we will add grounded versions that include source excerpts.

In [ ]:
PROMPTS = [
    "Summarize the typical structure of a US large-cap quarterly earnings call (prepared remarks vs Q&A).",
    "An analyst asks: what questions should I listen for when management discusses gross margin vs operating margin?",
    "Define 'guide' vs 'consensus' vs 'beat/miss' in an earnings-call context, in two sentences each.",
]

for i, p in enumerate(PROMPTS, 1):
    print("=" * 72)
    print(f"[{i}] {p}")
    print("-" * 72)
    print(harness.generate(p))
    print()

## 5. Done

If all three prompts returned coherent text, the baseline harness works.
Save this notebook output on Kaggle as the qualitative *base* snapshot.